In [18]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().parent

import pandas as pd
import csv

import src.constants as Con

In [19]:
from src.data_paths import (
    TEST_RUN_PATH
)

In [20]:
NEW_OUT_PATH = TEST_RUN_PATH / "csvs"

In [21]:
for xls_path in sorted(TEST_RUN_PATH.glob("*.xls")):        # source: parent folder
    df = pd.read_csv(
        xls_path,
        encoding="utf-16",
        sep="\t",
        quoting=csv.QUOTE_NONE,
        engine="python",
    )
    csv_path = NEW_OUT_PATH / xls_path.with_suffix(".csv").name   # dest: csvs/ folder
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"{xls_path.name}: {df.shape} -> {csv_path.name}")


fixations_answers.xls: (1344, 318) -> fixations_answers.csv
IA_answers.xls: (2162, 326) -> IA_answers.csv
messages_answers.xls: (577, 123) -> messages_answers.csv


In [22]:
fix_df = pd.read_csv(NEW_OUT_PATH / "fixations_answers.csv")
ia_df = pd.read_csv(NEW_OUT_PATH / "ia_answers.csv")
msg_df = pd.read_csv(NEW_OUT_PATH / "messages_answers.csv")

C:\Users\deeth\AppData\Local\Temp\ipykernel_28376\270841090.py:2: DtypeWarning: Columns (0: IA_FSA_COUNT_19, 1: IA_FSA_COUNT_20, 2: IA_FSA_COUNT_21, 3: IA_FSA_COUNT_22, 4: IA_FSA_COUNT_23, 5: IA_FSA_COUNT_24, 6: IA_FSA_COUNT_25, 7: IA_FSA_COUNT_26, 8: IA_FSA_COUNT_27, 9: IA_FSA_COUNT_28, 10: IA_FSA_COUNT_29, 11: IA_FSA_COUNT_30, 12: IA_FSA_COUNT_31, 13: IA_FSA_COUNT_32, 14: IA_FSA_COUNT_33, 15: IA_FSA_COUNT_34, 16: IA_FSA_COUNT_35, 17: IA_FSA_COUNT_36, 18: IA_FSA_COUNT_37, 19: IA_FSA_COUNT_38, 20: IA_FSA_COUNT_39, 21: IA_FSA_COUNT_40, 22: IA_FSA_DURATION_19, 23: IA_FSA_DURATION_20, 24: IA_FSA_DURATION_21, 25: IA_FSA_DURATION_22, 26: IA_FSA_DURATION_23, 27: IA_FSA_DURATION_24, 28: IA_FSA_DURATION_25, 29: IA_FSA_DURATION_26, 30: IA_FSA_DURATION_27, 31: IA_FSA_DURATION_28, 32: IA_FSA_DURATION_29, 33: IA_FSA_DURATION_30, 34: IA_FSA_DURATION_31, 35: IA_FSA_DURATION_32, 36: IA_FSA_DURATION_33, 37: IA_FSA_DURATION_34, 38: IA_FSA_DURATION_35, 39: IA_FSA_DURATION_36, 40: IA_FSA_DURATION_37, 41:

## Standardize new-data column names for the pipeline

The new experiment reports use different column names/indexing than the pipeline
expects. Rename them (data name → pipeline name), convert the numeric `answers_order`
to letters, and write cleaned CSVs to `csvs/cleaned/`, which the pipeline then reads:

| new data | pipeline expects |
|---|---|
| `RECORDING_SESSION_LABEL` | `participant_id` |
| `onestopqa_question_id` | `same_critical_span` |
| `practice` | `practice_trial` |
| `correct_answer` (0–3) | `correct_answer_position` |
| `FINAL_ANSWER` (0–3) | `selected_answer_position` |
| `answer_0..answer_3` (0-indexed) | `answer_1..answer_4` (1-indexed) |
| `answer_a..answer_d` | `answer_A..answer_D` |
| `answers_order` = `[2,1,0,3]` (screen→answer index) | `['C','B','A','D']` (letter per screen pos) |

Notes:
- `answer_A..D` are supplied directly, so the base feature that would recompute them
  (`add_answer_text_columns`) is excluded from the run.
- The new experiment has no repeated-reading trials (`repeated_reading_trial` column
  absent), so `remove_repeats` is turned off in the run below.

In [23]:
import ast

# Rename map: new-data column (left) -> pipeline column (right).
NEW_DATA_RENAME_MAP = {
    Con.RECORDING_SESSION_LABEL: Con.PARTICIPANT_ID,        # -> participant_id
    "onestopqa_question_id": Con.SAME_CRITICAL_SPAN_COLUMN,  # -> same_critical_span
    "practice": Con.PRACTICE_TRIAL_COLUMN,                   # -> practice_trial
    "correct_answer": Con.CORRECT_ANSWER_POSITION_COLUMN,    # -> correct_answer_position (0-3)
    "FINAL_ANSWER": Con.SELECTED_ANSWER_POSITION_COLUMN,     # -> selected_answer_position (0-3)
    # answers are 0-indexed (answer_0..answer_3); shift up one to the 1-indexed
    # answer_1..answer_4 the pipeline reads (this drops the old answer_0 name).
    f"{Con.ANSWER_PREFIX}0": f"{Con.ANSWER_PREFIX}1",
    f"{Con.ANSWER_PREFIX}1": f"{Con.ANSWER_PREFIX}2",
    f"{Con.ANSWER_PREFIX}2": f"{Con.ANSWER_PREFIX}3",
    f"{Con.ANSWER_PREFIX}3": f"{Con.ANSWER_PREFIX}4",
    # answer_a..answer_d already hold the labelled options; capitalize the letter
    # to the answer_A..answer_D the pipeline uses. Because these are supplied
    # directly, add_answer_text_columns (which would recompute them) is excluded
    # from the run below.
    f"{Con.ANSWER_PREFIX}a": f"{Con.ANSWER_PREFIX}A",
    f"{Con.ANSWER_PREFIX}b": f"{Con.ANSWER_PREFIX}B",
    f"{Con.ANSWER_PREFIX}c": f"{Con.ANSWER_PREFIX}C",
    f"{Con.ANSWER_PREFIX}d": f"{Con.ANSWER_PREFIX}D",
}


def _answers_order_to_letters(value):
    """Convert the new-data numeric answers_order to the letter form the pipeline reads.

    build_exp_inputs.py builds answers_order as screen_position -> original answer
    index (0=A, 1=B, 2=C, 3=D), e.g. [2, 1, 0, 3]. The pipeline expects the *letter*
    at each screen position, e.g. ['C', 'B', 'A', 'D']. Values already in letter form
    are left untouched.
    """
    order = ast.literal_eval(value) if isinstance(value, str) else value
    return [
        Con.ANSWER_LABELS[int(x)] if str(x).lstrip("-").isdigit() else x
        for x in order
    ]


def standardize_new_data_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Rename new-experiment columns to the names the pipeline expects and convert
    the numeric answers_order to letters.

    Renames are applied in a single pass (so the answer 0->1, 1->2, ... shift
    doesn't collide); columns not present are left untouched.
    """
    df = df.rename(columns=NEW_DATA_RENAME_MAP)
    if Con.ANSWERS_ORDER_COLUMN in df.columns:
        df[Con.ANSWERS_ORDER_COLUMN] = df[Con.ANSWERS_ORDER_COLUMN].apply(
            _answers_order_to_letters
        )
    return df


CLEANED_DIR = NEW_OUT_PATH / "cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

for _name in ["IA_answers.csv", "fixations_answers.csv"]:
    _df = pd.read_csv(NEW_OUT_PATH / _name, low_memory=False)
    _df = standardize_new_data_columns(_df)
    _out = CLEANED_DIR / _name
    _df.to_csv(_out, index=False)
    print(f"{_name}: {_df.shape} -> cleaned/{_name}")


IA_answers.csv: (2162, 326) -> cleaned/IA_answers.csv
fixations_answers.csv: (1344, 318) -> cleaned/fixations_answers.csv


## Run the `data_csv_generation` pipeline on the new experiment inputs

Skeleton for running the existing `src.data_prep.data_csv_generation.main()` pipeline
against the newly-converted IA + fixations CSVs (in `data_raw/testrun_QA/csvs/`) and
writing the result to `data/new_exp_try_runs/`.

**Not expected to run cleanly yet** — see the `TODO` notes in the function for the
known gaps (hardcoded fixation path in `create_fixation_sequence_tags`, button-clicks /
messages inputs, and possible column-name mismatches in the new raw format).

In [24]:
from src.data_prep import data_csv_generation as dcg
from src.data_paths import DATA_DIR

# --- New-experiment inputs (column-standardized CSVs from the cleaning cell) ---
NEW_IA_ANSWERS_PATH = NEW_OUT_PATH / "cleaned" / "IA_answers.csv"
NEW_FIX_ANSWERS_PATH = NEW_OUT_PATH / "cleaned" / "fixations_answers.csv"
NEW_MESSAGES_PATH = NEW_OUT_PATH / "messages_answers.csv"  # not standardized/used yet

# --- Where the processed output goes ---
NEW_EXP_OUT_DIR = DATA_DIR / "new_exp_try_runs"
NEW_EXP_OUT_PATH = NEW_EXP_OUT_DIR / "all_participants.csv"
NEW_EXP_AUX_DIR = NEW_EXP_OUT_DIR / "Auxiliary"

# Base features to skip for the new data. answer_A..D are already supplied by the
# cleaning step (renamed from answer_a..d), so we don't recompute them.
EXCLUDED_BASE_FUNCS = {"add_answer_text_columns"}


def run_new_exp_pipeline(
    ia_answers_path: Path = NEW_IA_ANSWERS_PATH,
    fixations_path: Path = NEW_FIX_ANSWERS_PATH,
    output_path: Path = NEW_EXP_OUT_PATH,
    aux_dir: Path = NEW_EXP_AUX_DIR,
    verbose: bool = True,
):
    """
    Run the data_csv_generation pipeline on the NEW experiment inputs and save the
    result under data/new_exp_try_runs/.

    Runs base + group features (auxiliary RT/last-label/button-clicks steps are off).
    """
    NEW_EXP_OUT_DIR.mkdir(parents=True, exist_ok=True)
    aux_dir.mkdir(parents=True, exist_ok=True)

    # Run all registered base features except the excluded ones (built from the
    # registry so it stays correct if base features are added/removed).
    base_function_names = [
        name
        for name, entry in dcg.FUNCTION_REGISTRY.items()
        if entry["kind"] == "base" and name not in EXCLUDED_BASE_FUNCS
    ]

    # `fixations_path` is the single canonical fixations report: it feeds both the
    # pupil stats and create_fixation_sequence_tags (group kwargs are forwarded to
    # the feature functions), so all registered group features can run.
    group_function_names = None  # None = all registered group features

    # TODO(new-data): button-clicks / RT / last-label features derive from the raw
    # messages report. Point button_clicks rebuild at NEW_MESSAGES_PATH (needs
    # plumbing in button_clicks_processing) or start with these off.
    add_last = False
    add_rts = False

    dcg.main(
        ia_answers_path=ia_answers_path,
        output_path=output_path,
        fixations_path=fixations_path,  # feeds pupil stats + fixation-sequence tags
        compute_pupil_stats=True,
        # keep every artifact inside the new-exp folder
        pupil_stats_path=aux_dir / "participant_pupils.csv",
        button_clicks_path=aux_dir / "button_clicks_data.csv",
        last_labels_path=aux_dir / "all_participants_last.csv",
        rt_and_tfd_path=aux_dir / "RT_and_TFD.csv",
        save_auxiliary=True,
        add_last=add_last,
        add_rts=add_rts,
        rebuild_button_clicks=False,  # TODO(new-data): rebuild from NEW_MESSAGES_PATH
        base_function_names=base_function_names,
        group_function_names=group_function_names,
        # the new experiment has no repeated-reading trials (no repeated_reading_trial
        # column), so the repeat filter is off; practice trials are still removed.
        remove_repeats=False,
        remove_practice=True,
        verbose=verbose,
    )

    if verbose:
        print(f"\n✓ New-exp pipeline output -> {output_path}")
    return output_path


run_new_exp_pipeline()



Building button-click data from raw (missing)…
Loading fix_csv from: E:\Technion\QA PROJECT\programming\QA_eyetracking_workspace\data_raw\full\fixations_Answers.csv


E:\Technion\QA PROJECT\programming\QA_eyetracking_workspace\src\data_prep\button_clicks_processing.py:424: DtypeWarning: Columns (0: CURRENT_FIX_NEAREST_INTEREST_AREA, 1: CURRENT_FIX_NEAREST_INTEREST_AREA_DISTANCE, 2: CURRENT_FIX_PRECISION_MEASURE_RMS_S2S, 3: CURRENT_FIX_PRECISION_MEASURE_SD_WINDOW) have mixed types. Specify dtype option on import or set low_memory=False.
  fix_csv = pd.read_csv(fix_csv_path)


Loading fix_tsv from: E:\Technion\QA PROJECT\programming\QA_eyetracking_workspace\data_raw\tsv\Fixations reports\fixations_A.tsv


E:\Technion\QA PROJECT\programming\QA_eyetracking_workspace\src\data_prep\button_clicks_processing.py:428: DtypeWarning: Columns (0: CURRENT_FIX_NEAREST_INTEREST_AREA, 1: CURRENT_FIX_NEAREST_INTEREST_AREA_DISTANCE, 2: CURRENT_FIX_PRECISION_MEASURE_RMS_S2S, 3: CURRENT_FIX_PRECISION_MEASURE_SD_WINDOW) have mixed types. Specify dtype option on import or set low_memory=False.
  fix_tsv = pd.read_csv(fix_tsv_path, sep="\t", encoding=fix_tsv_encoding)


Malformed trial rows: 19250
Affected recordings: 316
Rows: 718106 -> 152970

First malformed trial per affected recording:
    RECORDING_SESSION_LABEL  FIRST_MALFORMED_TRIAL
62                    l1_64                      1
111                 l28_242                      1
103                 l26_376                      2
34                   l15_95                      2
126                  l2_324                      2
162                 l36_280                      2
230                 l48_153                      2
272                 l56_159                      3
264                 l54_151                      3
63                  l20_101                      3
Saved trial_level_df to: E:\Technion\QA PROJECT\programming\QA_eyetracking_workspace\data\new_exp_try_runs\Auxiliary\button_clicks_data.csv
Final shape: (24046, 13)

Loading raw answers from: E:\Technion\QA PROJECT\programming\QA_eyetracking_workspace\data_raw\testrun_QA\csvs\cleaned\IA_answers.csv

Resolving proce

WindowsPath('E:/Technion/QA PROJECT/programming/QA_eyetracking_workspace/data/new_exp_try_runs/all_participants.csv')